In [1]:
# importing packages
import matplotlib.pyplot as plt
import ptitprince as pt
import random
import numpy as np
from numpy.random import default_rng
import pandas as pd
import json
from tqdm import tqdm
from scipy import stats, special
import seaborn as sns
import networkx as nx
import pickle
import itertools
from dotmotif import Motif, GrandIsoExecutor
from scipy.stats import kruskal, f_oneway, levene, ranksums, mannwhitneyu, ttest_ind, wilcoxon, norm, chi2_contingency, chisquare
from statsmodels.stats.multitest import multipletests
from sklearn import mixture
from scipy.interpolate import interp1d
from tabulate import tabulate
from statannotations.Annotator import Annotator
from typing import Tuple, Dict, Any
import warnings, time, random, logging
from pathlib import Path
from requests.exceptions import HTTPError
import caveclient
from pcg_skel import get_meshwork_from_client

plt.rcParams.update({'font.size': 20})
plt.rcParams["figure.figsize"] = (10,10)
sns.set_theme(style="whitegrid")
random.seed(747)

# Import Stefan's Library for Data Management of V1DD
from lsmm_data import LSMMData
warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)

# Pyramidal Rectangular Set

In [2]:
# Pull Data from LSMM Data
with open('pyr_cells_rectangular_connectome.json') as f:
    lsmm_json_input = json.load(f)
v1dd_data = LSMMData.LSMMData(lsmm_json_input)

data_a = v1dd_data.data
params_a = v1dd_data.params
dirs_a = v1dd_data.dirs
mappings_a = v1dd_data.mappings

Filtering to pyramidal cells only
459
Generating Connectome...


100%|██████████| 44293/44293 [03:40<00:00, 200.46it/s]


Generating Connectome...


100%|██████████| 44293/44293 [04:15<00:00, 173.25it/s]


In [3]:
### Pull necessary data from V1DD using LSMMData Manager
cell_table = data_a['structural']['pre_cell'].copy()
cell_table['connectome_index'] = cell_table.index
post_cell_table = data_a['structural']['post_cell'].copy()
post_cell_table['connectome_index'] = post_cell_table.index
synapse_table = data_a['structural']['synapse']

# Establish seperate sets for the pre and post synaptic partnes
# This is necessary as the set of connectome index of pre-synaptic cells do not 
# match the post-synaptic cells due to allowing unproofread post-synaptic targets.
individual_assembly_indexes = [mappings_a['connectome_indexes_by_assembly'][f'A {i}'] for i in range(1,16)]
individual_post_assembly_indexes = [mappings_a['post_connectome_indexes_by_assembly'][f'A {i}'] for i in range(1,16)]

coregistered_post_cell_indexes = mappings_a['assemblies_by_post_connectome_index'].keys()
coregistered_cell_indexes = mappings_a['assemblies_by_connectome_index'].keys()

no_a_cell_indexes = mappings_a['connectome_indexes_by_assembly']['No A']
no_a_post_cell_indexes = mappings_a['post_connectome_indexes_by_assembly']['No A']

pooled_assembly_indexes = list(set(coregistered_cell_indexes) - set(no_a_cell_indexes))
pooled_assembly_post_indexes = list(set(coregistered_post_cell_indexes) - set(no_a_post_cell_indexes))
# Each cell has a distinct root id, so it is unnecessary to establish different sets
assembly_to_root_ids = mappings_a['pt_root_ids_by_assembly']
assembly_root_ids_set = set(mappings_a['assemblies_by_pt_root_id'].keys())

# Filter synapses_table to only synapses between two assembly cells (including No A)
synapses_df = synapse_table[synapse_table['pre_pt_root_id'].isin(assembly_root_ids_set)]
synapses_df = synapses_df[synapses_df['post_pt_root_id'].isin(assembly_root_ids_set)]
synapses_df['size'] = synapses_df['size'] * (9 * 9 * 45) / (10**9) # Voxels -> Cubic micrometers

# Filter cell tables to only assembly cells
cell_table = cell_table[cell_table['pt_root_id'].isin(assembly_root_ids_set)]
post_cell_table = post_cell_table[post_cell_table['pt_root_id'].isin(assembly_root_ids_set)]

# Finalized set of Root IDs, which are 
pre_root_ids = set(cell_table['pt_root_id'].values)
post_root_ids = set(post_cell_table['pt_root_id'].values)
all_root_ids = pre_root_ids | post_root_ids

# Axonal Skeleton - Dendritic Spine Cotravel

# Save skeletons

In [5]:
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'int'):
    np.int = int

if not hasattr(np, "bool"):
    np.bool = bool 

# authorization
client = caveclient.CAVEclient(server_address='https://globalv1.em.brain.allentech.org', global_only=True)
client.auth.save_token(token="5a05953c19b5e120a3ac209f72df65ae", overwrite=True)
client = caveclient.CAVEclient('v1dd', server_address='https://globalv1.em.brain.allentech.org')

client.materialize.get_versions()
# materialization = 1290
materialization = 742
client.version = materialization

CACHE_DIR = Path("data_files/v1dd/structural/742/skeletons")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
timestamp = client.materialize.get_timestamp(742)
print(f"Timestamp for version {742} is {timestamp}")
nucleus_table = client.materialize.live_query('nucleus_detection_v0', timestamp=timestamp)

Timestamp for version 742 is 2024-05-20 08:10:01.264226+00:00


In [14]:
def cache_skeleton_true(root_id, client, version=4):
    """
    Fully MeshWork-faithful caching.
    Stores:
        - full skeleton (coords_full, edges_full)
        - axon skeleton (coords_ax, edges_ax)
        - dendrite skeleton (coords_de, edges_de)

    No mask projection.
    No attempt to align skeletons.
    Just exact MeshWork behavior, offline.
    """

    out_file = CACHE_DIR / f"{root_id}.npz"
    if out_file.exists():
        print(f"✔ Already cached: {root_id}")
        return

    print(f"\n⬇ Downloading true skeletons for: {root_id}")

    # -----------------------------
    # Load meshwork with skeleton
    # -----------------------------
    mw = get_meshwork_from_client(
        root_id,
        client=client,
        synapses=False,
        restore_graph=True,
        restore_properties=True,
        skeleton_version=version,
    )

    # -----------------------------
    # 1. FULL SKELETON
    # -----------------------------
    coords_full = mw.skeleton.vertices.astype(np.float32)
    edges_full  = mw.skeleton.edges.astype(np.int32)

    # -----------------------------
    # 2. AXON SKELETON
    # -----------------------------
    mesh_mask = getattr(mw.anno.is_axon, "mesh_mask", None)

    if mesh_mask is not None and mesh_mask.sum() > 0:
        mw.reset_mask()
        mw.apply_mask(mesh_mask)
        coords_ax = mw.skeleton.vertices.astype(np.float32)
        edges_ax  = mw.skeleton.edges.astype(np.int32)
    else:
        # No axon mask → empty axon skeleton
        coords_ax = np.zeros((0,3), dtype=np.float32)
        edges_ax  = np.zeros((0,2), dtype=np.int32)

    # -----------------------------
    # 3. DENDRITE SKELETON
    # -----------------------------
    mw.reset_mask()
    if mesh_mask is not None and mesh_mask.sum() > 0:
        dend_mask = ~mesh_mask
        mw.apply_mask(dend_mask)
        coords_de = mw.skeleton.vertices.astype(np.float32)
        edges_de  = mw.skeleton.edges.astype(np.int32)
    else:
        # No axon mask → dendrite is the entire cell
        coords_de = coords_full.copy()
        edges_de  = edges_full.copy()

    # -----------------------------
    # Save all to file
    # -----------------------------
    np.savez_compressed(
        out_file,
        coords_full=coords_full,
        edges_full=edges_full,
        coords_ax=coords_ax,
        edges_ax=edges_ax,
        coords_de=coords_de,
        edges_de=edges_de,
    )

    print(f"💾 Saved: {out_file}")

In [15]:
def cache_all_cells(root_ids, client):
    for rid in sorted(root_ids):
        try:
            cache_skeleton_true(rid, client)
        except Exception as e:
            print(f"❌ FAILED {rid}: {e}")

In [16]:
cache_all_cells(all_root_ids, client)


⬇ Downloading true skeletons for: 864691132534275418
💾 Saved: data_files\v1dd\structural\742\skeletons\864691132534275418.npz

⬇ Downloading true skeletons for: 864691132534315610
💾 Saved: data_files\v1dd\structural\742\skeletons\864691132534315610.npz

⬇ Downloading true skeletons for: 864691132548503618
💾 Saved: data_files\v1dd\structural\742\skeletons\864691132548503618.npz

⬇ Downloading true skeletons for: 864691132561381070
💾 Saved: data_files\v1dd\structural\742\skeletons\864691132561381070.npz

⬇ Downloading true skeletons for: 864691132562972388
💾 Saved: data_files\v1dd\structural\742\skeletons\864691132562972388.npz

⬇ Downloading true skeletons for: 864691132573738810
💾 Saved: data_files\v1dd\structural\742\skeletons\864691132573738810.npz

⬇ Downloading true skeletons for: 864691132574630714
💾 Saved: data_files\v1dd\structural\742\skeletons\864691132574630714.npz

⬇ Downloading true skeletons for: 864691132576897037
💾 Saved: data_files\v1dd\structural\742\skeletons\8646911

In [17]:
def build_cache_summary(pre_root_ids, post_root_ids, all_root_ids):
    rows = []

    for rid in sorted(all_root_ids):
        npz_path = CACHE_DIR / f"{rid}.npz"

        if not npz_path.exists():
            rows.append({
                "root_id": rid,
                "is_pre": rid in pre_root_ids,
                "is_post": rid in post_root_ids,
                "cached": False,
                "n_full_vertices": None,
                "n_axon_vertices": None,
                "n_dend_vertices": None,
                "has_axon_vertices": None,
                "has_dend_vertices": None,
            })
            continue

        data = np.load(npz_path)

        coords_full = data["coords_full"]
        coords_ax   = data["coords_ax"]
        coords_de   = data["coords_de"]

        rows.append({
            "root_id": rid,
            "is_pre": rid in pre_root_ids,
            "is_post": rid in post_root_ids,
            "cached": True,
            "n_full_vertices": coords_full.shape[0],
            "n_axon_vertices": coords_ax.shape[0],
            "n_dend_vertices": coords_de.shape[0],
            "has_axon_vertices": coords_ax.shape[0] > 0,
            "has_dend_vertices": coords_de.shape[0] > 0,
        })

    df = pd.DataFrame(rows)
    return df

In [18]:
summary_df = build_cache_summary(pre_root_ids, post_root_ids, all_root_ids)
summary_df
# summary_df['too_many_axons'] = summary_df['n_axon_vertices'] > summary_df['n_full_vertices']

,root_id,is_pre,is_post,cached,n_full_vertices,n_axon_vertices,n_dend_vertices,has_axon_vertices,has_dend_vertices
0,864691132534275418,True,True,True,3652,2505,1147,True,True
1,864691132534315610,True,True,True,3567,2449,1118,True,True
2,864691132548503618,True,True,True,4841,3244,1597,True,True
3,864691132561381070,False,True,True,1325,297,1028,True,True
4,864691132562972388,True,True,True,6485,4361,2124,True,True
...,...,...,...,...,...,...,...,...,...
310,864691133066738137,False,True,True,5420,3955,1465,True,True
311,864691133070374089,False,True,True,7561,5688,1873,True,True
312,864691133071145161,False,True,True,3027,1885,1142,True,True
313,864691133310618448,False,True,True,2203,0,2203,False,True
